[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# Proteina Complexa — Hands-On Notebook

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。

In [ ]:
#@title **ノートブックの表示設定**
from IPython.display import display, HTML
display(HTML("""
<style>
/* Limit output cell height and make scrollable */
.cell .output_wrapper .output {
    max-height: 200px;
    overflow-y: auto;
}
.output_scroll {
    max-height: 200px !important;
}
pre {
    max-height: 200px;
    overflow-y: auto;
}
</style>
"""))
print("Display settings applied.")

---
## 1. 環境構築

はじめに、必要なソフトウェアをインストールします。
以下のセルを **上から順に** 実行してください（各セルで Shift+Enter）。

### 1.1 GPU の確認

まず、GPUが割り当てられているか確認します。
「GPU が見つかりません」と表示された場合は、メニューの **「ランタイム」→「ランタイムのタイプを変更」** から
**GPU** を選択してください。

In [ ]:
#@title **GPU の確認**
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

### 1.2 パッケージマネージャ (uv) のインストールとリポジトリの取得

高速パッケージマネージャ **uv** をインストールし、Proteina Complexa のソースコードを取得します。

In [ ]:
#@title **uv のインストールとリポジトリの取得**
!pip install -q uv
!git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git
%cd Proteina-Complexa
print("Done.")

### 1.3 PyTorch のインストール

GPU対応版の PyTorch をインストールします。（数分かかります）

In [ ]:
#@title **PyTorch のインストール**
!uv pip install --system torch==2.7.0+cu126 torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 1.4 Proteina Complexa のインストール

本体と依存ライブラリをインストールします。（数分かかります）

In [ ]:
#@title **Proteina Complexa のインストール**
# Install base package
!uv pip install --system --index-strategy unsafe-best-match -e .

# Install PyTorch Geometric
!uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster \
    -f https://data.pyg.org/whl/torch-2.7.0+cu126.html

# Install Graphein and Atomworks
!uv pip install --system graphein==1.7.7 --no-deps
!uv pip install --system "atomworks[ml,openbabel,dev]" 2>/dev/null || echo "Warning: atomworks partial install"

# Install biotite
!uv pip install --system biotite==1.6.0

print("Done.")

### 1.5 構造予測モデル (Boltz-2) のインストール

生成したタンパク質の構造を予測・検証するためのモデルをインストールします。

In [ ]:
#@title **構造予測モデル (Boltz-2) のインストール**
!uv pip install --system "boltz[cuda]" -U

print("Done.")

### 1.6 モデルの重みをダウンロード

事前学習済みモデルをダウンロードします。（数分かかります）

In [ ]:
#@title **モデルの重みをダウンロード**
!complexa init
!complexa download --complexa-all

### 1.7 インストールの確認

全てのインストールが完了したか確認します。

In [ ]:
#@title **インストールの確認**
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!complexa --help > /dev/null 2>&1 && echo "Proteina Complexa CLI: OK" || echo "Proteina Complexa CLI: NOT FOUND"

print()
print("Environment setup complete!")